<a href="https://colab.research.google.com/github/BhumiPatil1507/BTP-/blob/main/Cp_MoE_Rigdestack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Cp EXPERT MoE + RIDGESTACK — END-TO-END COLAB PIPELINE
# Uses existing feature-extraction CSVs; does NOT regenerate features.
# Separate Cp 80/20 split, seed 42; 3-fold OOF stacking on train only.
# ============================================================

%pip -q install xgboost scikit-learn scipy joblib

import os, csv, gc, json, time, warnings
import numpy as np
import pandas as pd
import xgboost as xgb
import joblib

from google.colab import drive
from sklearn.model_selection import train_test_split, KFold
from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")
drive.mount("/content/drive")

# -------------------- CONFIGURATION --------------------
PROJECT_DIR = "/content/drive/MyDrive/Colab"
DATA_FILE = os.path.join(PROJECT_DIR, "polyOne_aa.csv")
FEATURE_DIR = os.path.join(PROJECT_DIR, "polymer_features")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "Cp_MOE_RIDGESTACK_results")
MODEL_DIR = os.path.join(OUTPUT_DIR, "models")
PRED_DIR = os.path.join(OUTPUT_DIR, "predictions")
OOF_DIR = os.path.join(OUTPUT_DIR, "oof")
SPLIT_DIR = os.path.join(OUTPUT_DIR, "splits")

for d in [OUTPUT_DIR, MODEL_DIR, PRED_DIR, OOF_DIR, SPLIT_DIR]:
    os.makedirs(d, exist_ok=True)

TARGET = "Cp"
SEED = 42
TEST_SIZE = 0.20
N_FOLDS = 3
CHUNK_SIZE = 5000
MAX_TREES = 2000
VALIDATION_SIZE = 0.10
EXPERTS = ["fingerprints", "descriptors", "graph", "functional_groups"]

FEATURE_FILES = {
    "fingerprints": os.path.join(FEATURE_DIR, "polyOne_aa_fingerprints.csv"),
    "descriptors": os.path.join(FEATURE_DIR, "polyOne_aa_descriptors.csv"),
    "graph": os.path.join(FEATURE_DIR, "polyOne_aa_graph.csv"),
    "functional_groups": os.path.join(FEATURE_DIR, "polyOne_aa_funcgroups.csv"),
}

for p in [DATA_FILE, *FEATURE_FILES.values()]:
    if not os.path.isfile(p):
        raise FileNotFoundError(f"Required file not found: {p}")

print("Target:", TARGET)
print("Base CSV:", DATA_FILE)
print("Output:", OUTPUT_DIR)
print("Feature files verified.")

# -------------------- TARGET + SPLIT --------------------
target_df = pd.read_csv(DATA_FILE, usecols=[TARGET], low_memory=False)
target_all = pd.to_numeric(target_df[TARGET], errors="coerce").to_numpy(np.float64)
del target_df
gc.collect()

valid_rows = np.flatnonzero(np.isfinite(target_all))
y = target_all[valid_rows].astype(np.float32)
positions = np.arange(len(y))

train_idx, test_idx = train_test_split(
    positions, test_size=TEST_SIZE, random_state=SEED, shuffle=True
)
y_train, y_test = y[train_idx], y[test_idx]
train_source_rows = valid_rows[train_idx]
test_source_rows = valid_rows[test_idx]

np.save(os.path.join(SPLIT_DIR, "Cp_valid_source_rows.npy"), valid_rows)
np.save(os.path.join(SPLIT_DIR, "Cp_train_indices.npy"), train_idx)
np.save(os.path.join(SPLIT_DIR, "Cp_test_indices.npy"), test_idx)
np.save(os.path.join(SPLIT_DIR, "Cp_train_source_rows.npy"), train_source_rows)
np.save(os.path.join(SPLIT_DIR, "Cp_test_source_rows.npy"), test_source_rows)

print(f"Rows with valid Cp: {len(y):,}")
print(f"Train rows: {len(train_idx):,} ({len(train_idx)/len(y):.1%})")
print(f"Test rows:  {len(test_idx):,} ({len(test_idx)/len(y):.1%})")
print("Cp range:", float(np.min(y)), "to", float(np.max(y)))

# -------------------- FEATURE COLUMN IDENTIFICATION --------------------
with open(DATA_FILE, "r", encoding="utf-8-sig", newline="") as f:
    base_columns = next(csv.reader(f))
base_set = set(base_columns)

FEATURE_COLUMNS = {}
for expert, path in FEATURE_FILES.items():
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        header = next(csv.reader(f))
    if TARGET not in header:
        raise ValueError(f"{TARGET} is absent from {path}; check feature extraction alignment.")
    cols = [c for c in header if c not in base_set]
    if not cols:
        raise ValueError(f"No extracted feature columns found for {expert}.")
    FEATURE_COLUMNS[expert] = cols
    print(f"{expert:20s}: {len(cols)} extracted features")

with open(os.path.join(OUTPUT_DIR, "Cp_feature_columns.json"), "w") as f:
    json.dump(FEATURE_COLUMNS, f, indent=2)

# -------------------- MEMORY-AWARE ROW LOADER --------------------
def load_feature_rows(expert, requested_source_rows):
    """Read selected original CSV rows from an aligned feature CSV, preserving order."""
    path = FEATURE_FILES[expert]
    cols = FEATURE_COLUMNS[expert]
    requested_source_rows = np.asarray(requested_source_rows, dtype=np.int64)

    if len(requested_source_rows) == 0:
        return np.empty((0, len(cols)), dtype=np.float32)

    order = np.argsort(requested_source_rows)
    sorted_rows = requested_source_rows[order]
    X_sorted = np.empty((len(sorted_rows), len(cols)), dtype=np.float32)

    left = 0
    csv_start = 0
    for chunk in pd.read_csv(path, usecols=cols, chunksize=CHUNK_SIZE, low_memory=False):
        csv_end = csv_start + len(chunk)
        right = np.searchsorted(sorted_rows, csv_end, side="left")
        if right > left:
            local = sorted_rows[left:right] - csv_start
            X_sorted[left:right] = chunk.iloc[local].to_numpy(dtype=np.float32, copy=True)
        left = right
        csv_start = csv_end
        del chunk
        if left >= len(sorted_rows):
            break

    if left != len(sorted_rows):
        raise ValueError(f"Could not retrieve all requested rows from {path}.")
    inverse = np.empty_like(order)
    inverse[order] = np.arange(len(order))
    X = X_sorted[inverse]
    del X_sorted
    gc.collect()

    if expert == "fingerprints":
        if not np.isfinite(X).all():
            raise ValueError("Fingerprint matrix contains missing/non-finite values.")
    else:
        # Imputation is fitted later using training data only.
        pass
    return X

# -------------------- TRAIN FOUR FINAL EXPERTS --------------------
# Each final expert uses an internal 10% validation subset for early stopping.
# Test set remains untouched until final evaluation.
final_test_predictions = []
expert_metrics = []

for expert in EXPERTS:
    print("\n" + "="*70)
    print("FINAL EXPERT:", expert, flush=True)
    started = time.time()

    model_path = os.path.join(MODEL_DIR, f"Cp_{expert}_expert.json")
    pred_path = os.path.join(PRED_DIR, f"Cp_{expert}_test_predictions.npy")
    metrics_path = os.path.join(MODEL_DIR, f"Cp_{expert}_metrics.json")
    imputer_path = os.path.join(MODEL_DIR, f"Cp_{expert}_imputer.joblib")

    X_train = load_feature_rows(expert, train_source_rows)
    X_test = load_feature_rows(expert, test_source_rows)

    imputer = None
    if expert != "fingerprints":
        imputer = SimpleImputer(strategy="median")
        X_train = imputer.fit_transform(X_train).astype(np.float32, copy=False)
        X_test = imputer.transform(X_test).astype(np.float32, copy=False)
        joblib.dump(imputer, imputer_path)

    if not np.isfinite(X_train).all() or not np.isfinite(X_test).all():
        raise ValueError(f"Non-finite values remain for {expert} after preprocessing.")

    inner_train, val_pos = train_test_split(
        np.arange(len(y_train)), test_size=VALIDATION_SIZE,
        random_state=SEED, shuffle=True
    )
    model = xgb.XGBRegressor(
        objective="reg:squarederror",
        n_estimators=MAX_TREES,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        tree_method="hist",
        eval_metric="rmse",
        early_stopping_rounds=100,
        n_jobs=-1,
        random_state=SEED,
    )
    model.fit(X_train[inner_train], y_train[inner_train],
              eval_set=[(X_train[val_pos], y_train[val_pos])], verbose=100)

    pred = model.predict(X_test).astype(np.float64)
    metrics = {
        "target": TARGET, "expert": expert, "feature_count": len(FEATURE_COLUMNS[expert]),
        "R2": float(r2_score(y_test, pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_test, pred))),
        "MAE": float(mean_absolute_error(y_test, pred)),
        "best_iteration": int(model.best_iteration),
        "trees_used": int(model.best_iteration + 1),
        "training_minutes": float((time.time()-started)/60),
    }
    model.save_model(model_path)
    np.save(pred_path, pred)
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=2)
    final_test_predictions.append(pred)
    expert_metrics.append(metrics)
    print(json.dumps(metrics, indent=2), flush=True)

    del X_train, X_test, model, imputer
    gc.collect()

test_pred_matrix = np.column_stack(final_test_predictions).astype(np.float64)
np.save(os.path.join(PRED_DIR, "Cp_all_expert_test_predictions.npy"), test_pred_matrix)
pd.DataFrame(expert_metrics).to_csv(
    os.path.join(OUTPUT_DIR, "Cp_individual_expert_metrics.csv"), index=False
)

# -------------------- BUILD OOF PREDICTIONS FOR STACKING --------------------
# OOF rows are in exactly the order of train_idx. Each row is predicted by
# an expert that did not train on that row. These predictions train both
# the ExtraTrees MoE router and the RidgeStack without test leakage.
OOF_PATH = os.path.join(OOF_DIR, "Cp_oof_predictions.npy")
OOF_PROGRESS = os.path.join(OOF_DIR, "Cp_oof_progress.json")
n_train = len(train_idx)
oof = np.lib.format.open_memmap(
    OOF_PATH, mode="w+", dtype=np.float32, shape=(n_train, len(EXPERTS))
)
oof[:] = np.nan
oof.flush()

# Match OOF model tree counts to the final experts' selected tree counts.
tree_counts = {}
for expert in EXPERTS:
    m = xgb.XGBRegressor()
    m.load_model(os.path.join(MODEL_DIR, f"Cp_{expert}_expert.json"))
    tree_counts[expert] = max(1, int(m.get_booster().num_boosted_rounds()))
    del m
print("\nOOF tree counts:", tree_counts)

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
completed = []
for fold, (fit_pos, hold_pos) in enumerate(kf.split(np.arange(n_train)), start=1):
    print(f"\nOOF fold {fold}/{N_FOLDS}: fit={len(fit_pos):,}, holdout={len(hold_pos):,}", flush=True)
    fit_source = train_source_rows[fit_pos]
    hold_source = train_source_rows[hold_pos]
    y_fit = y_train[fit_pos]

    for j, expert in enumerate(EXPERTS):
        print(f"  OOF expert: {expert}", flush=True)
        X_fit = load_feature_rows(expert, fit_source)
        X_hold = load_feature_rows(expert, hold_source)

        if expert != "fingerprints":
            imp = SimpleImputer(strategy="median")
            X_fit = imp.fit_transform(X_fit).astype(np.float32, copy=False)
            X_hold = imp.transform(X_hold).astype(np.float32, copy=False)

        model = xgb.XGBRegressor(
            n_estimators=tree_counts[expert],
            max_depth=8,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            tree_method="hist",
            objective="reg:squarederror",
            n_jobs=-1,
            random_state=SEED + fold,
            verbosity=0,
        )
        model.fit(X_fit, y_fit, verbose=False)
        oof[hold_pos, j] = model.predict(X_hold).astype(np.float32)
        oof.flush()
        completed.append(f"{expert}_fold_{fold}")
        with open(OOF_PROGRESS, "w") as f:
            json.dump({"target": TARGET, "folds": N_FOLDS, "completed": completed}, f, indent=2)

        del X_fit, X_hold, model
        if expert != "fingerprints":
            del imp
        gc.collect()

    del fit_source, hold_source, y_fit
    gc.collect()

oof.flush()
if not np.isfinite(oof).all():
    raise ValueError("OOF predictions contain non-finite values. Do not train stackers.")
print("\nOOF completed:", oof.shape, "| tasks:", len(completed), "/", N_FOLDS*len(EXPERTS))
del oof
gc.collect()

# -------------------- TRAIN MoE ROUTER + RIDGESTACK --------------------
oof_matrix = np.load(OOF_PATH, mmap_mode="r")
if oof_matrix.shape != (len(y_train), len(EXPERTS)):
    raise ValueError(f"Unexpected OOF shape {oof_matrix.shape}")
if not np.isfinite(oof_matrix).all():
    raise ValueError("OOF matrix is not finite.")

# MoE: nonlinear ExtraTrees router trained on out-of-fold expert predictions.
router = ExtraTreesRegressor(
    n_estimators=300, max_depth=12, min_samples_leaf=20,
    max_features=1.0, n_jobs=-1, random_state=SEED
)
router.fit(np.asarray(oof_matrix), y_train)
joblib.dump(router, os.path.join(MODEL_DIR, "Cp_MoE_router.joblib"))
moe_pred = router.predict(test_pred_matrix)

# RidgeStack: regularized linear stacking model trained on the SAME OOF matrix.
# RidgeCV chooses alpha by internal CV using only training OOF predictions.
ridge_stack = RidgeCV(alphas=np.logspace(-4, 4, 25), cv=5)
ridge_stack.fit(np.asarray(oof_matrix), y_train)
joblib.dump(ridge_stack, os.path.join(MODEL_DIR, "Cp_RidgeStack.joblib"))
ridge_pred = ridge_stack.predict(test_pred_matrix)

# -------------------- METRICS + DISAGREEMENT UNCERTAINTY --------------------
def metrics_row(name, truth, pred):
    return {
        "Model": name,
        "R2": float(r2_score(truth, pred)),
        "RMSE": float(np.sqrt(mean_squared_error(truth, pred))),
        "MAE": float(mean_absolute_error(truth, pred)),
    }

rows = [metrics_row(expert, y_test, test_pred_matrix[:, i])
        for i, expert in enumerate(EXPERTS)]
rows.append(metrics_row("MoE_ExtraTrees", y_test, moe_pred))
rows.append(metrics_row("RidgeStack", y_test, ridge_pred))
metrics_df = pd.DataFrame(rows)
metrics_path = os.path.join(OUTPUT_DIR, "Cp_experts_MoE_RidgeStack_metrics.csv")
metrics_df.to_csv(metrics_path, index=False)

# Di = sample standard deviation across the four individual expert predictions.
Di = np.std(test_pred_matrix, axis=1, ddof=1)
moe_abs_error = np.abs(y_test - moe_pred)
ridge_abs_error = np.abs(y_test - ridge_pred)

def safe_corr(a, b):
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])

uncertainty_df = pd.DataFrame({
    "Source_Row": test_source_rows,
    "Actual_Cp": y_test,
    "MoE_Predicted_Cp": moe_pred,
    "MoE_Absolute_Error": moe_abs_error,
    "RidgeStack_Predicted_Cp": ridge_pred,
    "RidgeStack_Absolute_Error": ridge_abs_error,
    "Disagreement_Di": Di,
})
for i, expert in enumerate(EXPERTS):
    uncertainty_df[f"{expert}_Prediction"] = test_pred_matrix[:, i]

uncertainty_path = os.path.join(PRED_DIR, "Cp_test_predictions_Di_and_errors.csv")
uncertainty_df.to_csv(uncertainty_path, index=False)

summary = {
    "target": TARGET,
    "n_valid": int(len(y)),
    "n_train": int(len(y_train)),
    "n_test": int(len(y_test)),
    "split_seed": SEED,
    "test_size": TEST_SIZE,
    "n_oof_folds": N_FOLDS,
    "ridge_alpha_selected": float(ridge_stack.alpha_),
    "Di_mean": float(np.mean(Di)),
    "Di_median": float(np.median(Di)),
    "Di_p95": float(np.percentile(Di, 95)),
    "corr_Di_MoE_absolute_error": safe_corr(Di, moe_abs_error),
    "corr_Di_RidgeStack_absolute_error": safe_corr(Di, ridge_abs_error),
}
with open(os.path.join(OUTPUT_DIR, "Cp_run_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "="*80)
print("Cp: INDIVIDUAL EXPERTS vs MoE vs RIDGESTACK — UNTOUCHED TEST SET")
print("="*80)
print(metrics_df.to_string(index=False, float_format=lambda v: f"{v:.6f}"))
print("\nRidgeStack selected alpha:", ridge_stack.alpha_)
print("\nDISAGREEMENT UNCERTAINTY (Di = std dev across 4 expert predictions)")
print(f"Mean Di: {summary['Di_mean']:.6f}")
print(f"Median Di: {summary['Di_median']:.6f}")
print(f"95th percentile Di: {summary['Di_p95']:.6f}")
print(f"Correlation Di vs MoE absolute error: {summary['corr_Di_MoE_absolute_error']:.6f}")
print(f"Correlation Di vs RidgeStack absolute error: {summary['corr_Di_RidgeStack_absolute_error']:.6f}")
print("\nSaved outputs:")
print("Metrics:", metrics_path)
print("Predictions + Di:", uncertainty_path)
print("Models:", MODEL_DIR)
print("OOF:", OOF_PATH)
print("\nDONE.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Target: Cp
Base CSV: /content/drive/MyDrive/Colab/polyOne_aa.csv
Output: /content/drive/MyDrive/Colab/Cp_MOE_RIDGESTACK_results
Feature files verified.
Rows with valid Cp: 500,000
Train rows: 400,000 (80.0%)
Test rows:  100,000 (20.0%)
Cp range: 0.8982343077659607 to 2.2664685249328613
fingerprints        : 2048 extracted features
descriptors         : 16 extracted features
graph               : 17 extracted features
functional_groups   : 33 extracted features

FINAL EXPERT: fingerprints
[0]	validation_0-rmse:0.11093
[100]	validation_0-rmse:0.06209
[200]	validation_0-rmse:0.05584
[300]	validation_0-rmse:0.05276
[400]	validation_0-rmse:0.05086
[500]	validation_0-rmse:0.04933
[600]	validation_0-rmse:0.04812
[700]	validation_0-rmse:0.04717
[800]	validation_0-rmse:0.04631
[900]	validation_0-rmse:0.04562
[1000]	validation_0-rmse:0.04504
[1100]	validation_0-rmse:0.